In [ ]:
import os
from pathlib import Path
from types import SimpleNamespace

import numpy as np

try:
    from scipy import ndimage, signal, optimize, interpolate, stats
except Exception:
    ndimage = signal = optimize = interpolate = stats = None

def n_elements(x):
    if x is None:
        return 0
    try:
        return np.size(x)
    except Exception:
        return 1

def read_parameter_file(parfile):
    params = {}
    path = Path(parfile)
    if not path.exists():
        raise FileNotFoundError(parfile)

    for raw in path.read_text().splitlines():
        line = raw.split(';', 1)[0].strip()
        if not line or '=' not in line:
            continue
        key, value = line.split('=', 1)
        key = key.strip().lower()
        value = value.strip()
        value = value.replace('[', 'np.array([').replace(']', '])')
        value = value.replace('^', '**')
        try:
            params[key] = eval(value, {'np': np, 'array': np.array})
        except Exception:
            params[key] = value.strip("'\"")
    return params

def write_idl_array_line(f, name, arr, comment=''):
    arr = np.asarray(arr).ravel()
    values = ','.join(f'{v:10.4E}' if abs(v) >= 1e4 or (abs(v) < 1e-3 and v != 0) else f'{v:10.4f}' for v in arr)
    f.write(f'{name.upper()}=[{values}]')
    if comment:
        f.write(f' ; {comment}')
    f.write('\n')

def robust_sigma(values):
    values = np.asarray(values, dtype=float)
    med = np.nanmedian(values)
    return 1.4826 * np.nanmedian(np.abs(values - med))

def linear_interp(x, y, x_new):
    return np.interp(x_new, np.asarray(x, dtype=float), np.asarray(y, dtype=float))

def congrid(array, new_shape):
    array = np.asarray(array, dtype=float)
    if ndimage is None:
        raise ImportError('scipy is required for congrid')
    zoom = [n / o for n, o in zip(new_shape, array.shape)]
    return ndimage.zoom(array, zoom, order=1)

def idl_hist2d(x, y, xbin, ybin, xmin, xmax, ymin, ymax):
    x_edges = np.arange(xmin, xmax + xbin, xbin)
    y_edges = np.arange(ymin, ymax + ybin, ybin)
    hist, _, _ = np.histogram2d(x, y, bins=[x_edges, y_edges])
    return hist

def load_model_file(path):
    with open(path, 'r') as f:
        age_line = f.readline().strip()
        feh_line = f.readline().strip()
        shape_line = f.readline().strip()
        shape = tuple(int(v) for v in shape_line.replace(',', ' ').split()[:2])
        data = np.loadtxt(f)
    return age_line, feh_line, data.reshape(shape)

def save_model_file(path, age_label, feh_label, model):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    model = np.asarray(model, dtype=float)
    with open(path, 'w') as f:
        f.write(age_label + '\n')
        f.write(feh_label + '\n')
        f.write(f'{model.shape[0]} {model.shape[1]}\n')
        np.savetxt(f, model)


In [ ]:
def grid_points(x, y, xmin, xmax, xbin, ymin, ymax, ybin):
    h2 = np.zeros((int((xmax - xmin) / xbin + 1), int((ymax - ymin) / ybin + 1)))
    use = (x >= xmin) & (x <= xmax) & (y >= ymin) & (y <= ymax)
    xel = ((x[use] - xmin) / xbin).astype(int)
    yel = ((y[use] - ymin) / ybin).astype(int)
    np.add.at(h2, (xel, yel), 1.0)
    return h2

def sfrinit(parfile, fileout):
    p = read_parameter_file(parfile)
    name = p.get('name', 'model')
    x = np.asarray(p['x'], dtype=float)
    y = np.asarray(p['y'], dtype=float)
    agebin = np.asarray(p['agebin'], dtype=float)
    agesz = np.asarray(p['agesz'], dtype=float)
    fehbin = np.asarray(p['fehbin'], dtype=float)
    sfr0 = np.asarray(p.get('sfr0', np.ones(len(agebin))), dtype=float)
    scl = np.asarray(p.get('scl', np.ones(len(agebin))), dtype=float)

    h2 = grid_points(x, y, p['xmin'], p['xmax'], p['xbin'], p['ymin'], p['ymax'], p['ybin'])
    models = []
    for ii in range(len(agebin)):
        path = Path('models') / f'{name}{ii}.dat'
        if path.exists():
            _, _, model = load_model_file(path)
            if model.shape != h2.shape:
                raise ValueError('Incompatible model and observation grids')
            models.append(model)

    h2mod = np.stack(models, axis=0)
    c = h2mod.shape[0]
    h2o = h2.ravel()
    h2msub = h2mod.reshape(c, -1)

    sfr, fitpar = sfrsolve(h2o, h2msub, sfr0[:c], scl[:c], p.get('partype'))
    for _ in range(5):
        sfr, fitpar = sfrsolve(h2o, h2msub, np.maximum(sfr, 0), np.abs(sfr), p.get('partype'))
    sfr = np.maximum(sfr, 0)

    h2d = np.sum(h2mod * sfr[:, None, None], axis=0)
    with open(fileout, 'w') as f:
        f.write(f"model = '{name}'\n")
        write_idl_array_line(f, 'SFR', sfr, 'M_sun yr^-1')
        write_idl_array_line(f, 'AGE', agebin[:c], 'Gyr')
        write_idl_array_line(f, 'AGESZ', agesz[:c], 'Gyr')
        write_idl_array_line(f, 'FEH', fehbin[:c], 'dex')
        f.write(f'FIT_VALUE={fitpar[0]:10.4f}\n')
        f.write(f'NDEG={np.count_nonzero(h2o > 0) - np.count_nonzero(sfr > 0)} ; degrees of freedom\n')
    return sfr, agebin[:c], agesz[:c], fehbin[:c], h2, h2d, h2mod
